In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

INPUT_PATH  = "../../../data/phase2/features_for_model.parquet"
OUTPUT_PATH = "../../../data/phase2/validation_results.parquet"

FEATURE_COLS = [
    "ema_ratio", "rsi_14", "macd_hist", "atr_14",
    "session_quality_enc", "direction_enc", "signal_valid_enc",
]
LABEL_COL = "label"

In [2]:
def train_fold(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
) -> tuple[np.ndarray, StandardScaler, LogisticRegression]:
    """
    Fit StandardScaler + LogisticRegression on training data.
    Return (predicted_probabilities_for_test, fitted_scaler, fitted_model).

    Scaler is fit only on X_train to prevent data leakage.
    LogisticRegression uses C=1.0, max_iter=1000, solver='lbfgs'.
    Returns probabilities for the positive class (label=1).
    """
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    model = LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", random_state=42)
    model.fit(X_train_scaled, y_train)

    proba = model.predict_proba(X_test_scaled)[:, 1]  # P(label=1)
    return proba, scaler, model


def run_walk_forward(df: pd.DataFrame) -> pd.DataFrame:
    """
    Run walk-forward validation across all folds.

    For each fold N:
      1. Filter train rows (fold==N, split=="train")
      2. Filter test rows  (fold==N, split=="test")
      3. Call train_fold
      4. Append test rows + predicted probability to results

    Returns DataFrame with columns:
      date, s3_key, fold, label, prob_itm, signal_valid_enc, direction_enc
    """
    folds = sorted(df[df["split"] == "test"]["fold"].unique())
    results = []

    for fold_idx in folds:
        train = df[(df["fold"] == fold_idx) & (df["split"] == "train")]
        test  = df[(df["fold"] == fold_idx) & (df["split"] == "test")]

        if len(train) < 10:
            print(f"Fold {fold_idx}: skipping — only {len(train)} train rows")
            continue
        if len(test) == 0:
            print(f"Fold {fold_idx}: skipping — no test rows")
            continue

        X_train = train[FEATURE_COLS]
        y_train = train[LABEL_COL]
        X_test  = test[FEATURE_COLS]

        proba, _, model = train_fold(X_train, y_train, X_test)

        fold_results = test[["date", "s3_key", "fold", LABEL_COL, "signal_valid_enc", "direction_enc"]].copy()
        fold_results["prob_itm"] = proba

        n_pos = int((y_train == 1).sum())
        n_neg = int((y_train == 0).sum())
        print(f"Fold {fold_idx}: train={len(train)} (pos={n_pos}, neg={n_neg}), test={len(test)}, coef={model.coef_[0].round(3)}")
        results.append(fold_results)

    if not results:
        raise RuntimeError("No folds produced results.")
    return pd.concat(results, ignore_index=True)

In [3]:
df = pd.read_parquet(INPUT_PATH)
results = run_walk_forward(df)

print(f"\nTotal test predictions: {len(results)}")
print(f"prob_itm range: {results['prob_itm'].min():.3f} – {results['prob_itm'].max():.3f}")
print(f"Label distribution in test set:\n{results['label'].value_counts()}")

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
results.to_parquet(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")

Fold 0: train=114 (pos=49, neg=65), test=140, coef=[-0.756 -0.813 -0.89  -0.121  0.818 -0.569 -0.271]
Fold 1: train=175 (pos=139, neg=36), test=161, coef=[ 0.178  0.334  0.241  0.439 -0.026 -1.264 -0.344]
Fold 2: train=124 (pos=42, neg=82), test=194, coef=[-0.881 -0.65  -0.477  0.079  0.343 -1.724 -0.099]
Fold 3: train=237 (pos=96, neg=141), test=86, coef=[-0.533 -0.024 -1.193  0.305  0.522 -1.774 -0.492]
Fold 4: train=151 (pos=29, neg=122), test=141, coef=[-0.767 -0.812 -0.339 -0.356 -0.216 -1.708  0.562]
Fold 5: train=207 (pos=57, neg=150), test=166, coef=[-1.293  0.192 -0.71  -0.586  0.157 -0.847 -0.237]
Fold 6: skipping — only 0 train rows
Fold 7: skipping — only 0 train rows
Fold 8: skipping — only 0 train rows
Fold 9: skipping — only 0 train rows
Fold 10: skipping — only 0 train rows
Fold 11: skipping — only 0 train rows
Fold 12: skipping — only 0 train rows
Fold 13: skipping — only 0 train rows
Fold 14: skipping — only 0 train rows
Fold 15: skipping — only 0 train rows
Fold 16: 

In [4]:
df = pd.read_parquet(OUTPUT_PATH)
print(df[["date", "s3_key", "fold", "label", "prob_itm"]].head(20).to_string())

                        date  s3_key  fold  label  prob_itm
0  2020-01-02 00:00:00+00:00  EURUSD     0    0.0  0.059741
1  2020-01-03 00:00:00+00:00  EURUSD     0    0.0  0.104672
2  2020-01-06 00:00:00+00:00  EURUSD     0    0.0  0.114944
3  2020-01-07 00:00:00+00:00  EURUSD     0    0.0  0.087095
4  2020-01-08 00:00:00+00:00  EURUSD     0    0.0  0.132001
5  2020-01-28 00:00:00+00:00  EURUSD     0    1.0  0.646592
6  2020-01-29 00:00:00+00:00  EURUSD     0    1.0  0.641182
7  2020-01-30 00:00:00+00:00  EURUSD     0    1.0  0.656225
8  2020-01-31 00:00:00+00:00  EURUSD     0    1.0  0.612622
9  2020-01-02 00:00:00+00:00  GBPUSD     0    0.0  0.058959
10 2020-01-03 00:00:00+00:00  GBPUSD     0    0.0  0.090416
11 2020-01-24 00:00:00+00:00  GBPUSD     0    0.0  0.119457
12 2020-01-13 00:00:00+00:00  USDJPY     0    0.0  0.303145
13 2020-01-14 00:00:00+00:00  USDJPY     0    0.0  0.243007
14 2020-01-15 00:00:00+00:00  USDJPY     0    0.0  0.265763
15 2020-01-16 00:00:00+00:00  USDJPY    